In [2]:
from pathlib import Path
from bs4 import BeautifulSoup
import re
import pandas as pd

In [3]:
# Adjust the filename to match what you saved:
raw_path = Path("../data/raw/BDHSC_2024_15255_math_advanced.html")
html = raw_path.read_text(encoding="utf-8")
print(html[:500])   # sanity check: should look like HTML

<!DOCTYPE html>
<!-- saved from url=(0074)https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12_15255.html -->
<html xmlns="http://www.w3.org/1999/xhtml" lang="en-AU" class="js"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
	
    <meta name="viewport" content="width=device-width, initial-scale=1">
    
		    
	<meta name="DC.Identifier" scheme="URI" content="http://www.boardofstudies.nsw.edu.au">     
	<meta name="DC.Title" content="Mathematics Advanced 2 un


In [4]:
%pip install lxml

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install lxml
soup = BeautifulSoup(html, "lxml")

Note: you may need to restart the kernel to use updated packages.


In [6]:
soup = BeautifulSoup(html, "html.parser")

In [7]:
def parse_course_page(html, year, course_name, course_code):
    """Parse one HSC course band page into a list of row dicts."""
    soup = BeautifulSoup(html, "html.parser")

    # candidature (your existing code)
    page_text = soup.get_text().replace("\xa0", " ")
    candidature = int(
        re.search(r"Candidature\s*-\s*([\d,]+)", page_text).group(1).replace(",", "")
    )

    rows = []
    for s in soup.find_all("strong"):
        label_text = s.get_text().replace("\xa0", " ").strip()
        if not label_text.startswith("Band"):
            continue
        cell_text = s.parent.get_text().replace("\xa0", " ")
        band = re.search(r"Band\s+(\S+)", label_text).group(1)
        percentage = float(re.search(r"\(([\d.]+)%\)", cell_text).group(1))

        # TODO: append a dict with these keys:
        #   band, percentage, candidature, year, course_name, course_code
        
        rows.append({
            "band" : band,
            "percentage" : percentage,
            "candidature" : candidature,
            "year" : year,
            "course_name" : course_name,
            "course_code" : course_code
        })


    return rows

In [8]:
import requests
url = "https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12_15255.html"
headers = {"User-Agent": "Mozilla/5.0 (HSC learning project)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()
print(resp.status_code, len(resp.text))

200 15114


In [9]:
rows = parse_course_page(resp.text, year=2024,
                         course_name="Mathematics Advanced",
                         course_code="15255")
pd.DataFrame(rows)

,band,percentage,candidature,year,course_name,course_code
0,6,22.33,16561,2024,Mathematics Advanced,15255
1,5,27.70,16561,2024,Mathematics Advanced,15255
2,4,27.32,16561,2024,Mathematics Advanced,15255
3,3,17.40,16561,2024,Mathematics Advanced,15255
4,2,4.71,16561,2024,Mathematics Advanced,15255
5,1,0.54,16561,2024,Mathematics Advanced,15255


In [10]:
index_url = "https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12.html"
resp = requests.get(index_url, headers=headers, timeout=30)
resp.raise_for_status()
index_soup = BeautifulSoup(resp.text, "html.parser")

course_links = index_soup.find_all("a", href=re.compile(r"BDHSC_\d+_12_\d+\.html"))
print(len(course_links))                       # expect ~150
for a in course_links[:5]:
    print(a["href"], "|", a.get_text(strip=True))

95
BDHSC_2024_12_15000.html | Aboriginal Studies 2 unit (15000)
BDHSC_2024_12_15010.html | Agriculture 2 unit (15010)
BDHSC_2024_12_15020.html | Ancient History 2 unit (15020)
BDHSC_2024_12_15510.html | Arabic Continuers 2 unit (15510)
BDHSC_2024_12_15520.html | Arabic Extension 1 unit (15520)


In [11]:
print(course_links[0])

<a href="BDHSC_2024_12_15000.html">Aboriginal Studies 2 unit (15000)</a>


In [12]:
courses = []
for a in course_links:
    href = a["href"]
    text = a.get_text(strip=True)       # e.g. "Biology 2 unit (15030)"

    # TODO 1: pull the course_code out of href -> "15030"
    #   hint: re.search(r"_(\d+)\.html", href).group(1)
    course_code = re.search(r"_(\d+)\.html",href).group(1)

    # TODO 2: pull a clean course_name out of text -> "Biology"
    #   hint: the name is everything before " N unit ..."
    #         try re.match(r"(.+?)\s+\d+\s*unit", text).group(1)
    course_name = re.match(r"(.+?)\s+\d+\s*unit",text).group(1)

    courses.append({"course_code": course_code, "course_name": course_name})

index_df = pd.DataFrame(courses)
index_df

,course_code,course_name
0,15000,Aboriginal Studies
1,15010,Agriculture
2,15020,Ancient History
3,15510,Arabic Continuers
4,15520,Arabic Extension
...,...,...
90,15390,Textiles and Design
91,27499,"Tourism, Travel and Events Examination"
92,16120,Turkish Continuers
93,16140,Vietnamese Continuers


In [13]:
targets = [
    "Mathematics Advanced",
    "Mathematics Extension 1",
    "Mathematics Extension 2",
    "English Advanced",
    "English Standard",
    "Biology",
    "Chemistry",
    "Physics",
    "Legal Studies",
]

target_df = index_df[index_df["course_name"].isin(targets)]
target_df

,course_code,course_name
6,15030,Biology
9,15050,Chemistry
24,15140,English Advanced
28,15130,English Standard
61,15220,Legal Studies
62,15255,Mathematics Advanced
63,15250,Mathematics Extension 1
64,15260,Mathematics Extension 2
75,15330,Physics


In [14]:
print(len(target_df), "of", len(targets))

9 of 9


In [15]:
course_url = "https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12_15030.html"

resp = requests.get(course_url, headers=headers, timeout=30)
resp.raise_for_status()
print(resp.status_code)
for course in target_df.itertuples():
    print(course.course_name)

200
Biology
Chemistry
English Advanced
English Standard
Legal Studies
Mathematics Advanced
Mathematics Extension 1
Mathematics Extension 2
Physics


In [16]:
import time

all_rows = []
for course in target_df.itertuples():
    course_url = f"https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12_{course.course_code}.html"

    resp = requests.get(course_url, headers=headers, timeout=30)
    resp.raise_for_status()

    # TODO: call parse_course_page(...) with resp.text and the right metadata
    #   (year=2024, and course.course_name / course.course_code from the loop)
    #   then all_rows.extend(...) the result
    res = parse_course_page(resp.text,year=2024,course_name=course.course_name,course_code=course.course_code)
    all_rows.extend(res)

    print("fetched", course.course_name)
    time.sleep(1)          # be polite — 1 second between requests

df_2024 = pd.DataFrame(all_rows)
df_2024

fetched Biology
fetched Chemistry
fetched English Advanced
fetched English Standard
fetched Legal Studies
fetched Mathematics Advanced
fetched Mathematics Extension 1
fetched Mathematics Extension 2
fetched Physics


,band,percentage,candidature,year,course_name,course_code
0,6,6.70,19046,2024,Biology,15030
1,5,28.89,19046,2024,Biology,15030
2,4,34.51,19046,2024,Biology,15030
3,3,18.14,19046,2024,Biology,15030
4,2,9.53,19046,2024,Biology,15030
5,1,2.22,19046,2024,Biology,15030
6,6,11.45,9722,2024,Chemistry,15050
7,5,27.37,9722,2024,Chemistry,15050
8,4,28.44,9722,2024,Chemistry,15050
9,3,20.00,9722,2024,Chemistry,15050


In [17]:
from pathlib import Path
import time

PAGES = Path("../data/raw/pages")
PAGES.mkdir(parents=True, exist_ok=True)

def get_html(url, filename):
    """Fetch url, caching to data/raw/pages/. Returns local copy if it already exists."""
    path = PAGES / filename
    if path.exists():
        return path.read_text(encoding="utf-8")          # cache hit — no network
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    path.write_text(resp.text, encoding="utf-8")          # save for next time
    time.sleep(1)                                          # polite pause, only on real fetch
    return resp.text

In [18]:
BASE = "https://www.boardofstudies.nsw.edu.au/ebos/static"

def get_target_courses(year):
    html = get_html(f"{BASE}/BDHSC_{year}_12.html", f"index_{year}.html")
    soup = BeautifulSoup(html, "html.parser")
    links = soup.find_all("a", href=re.compile(r"BDHSC_\d+_12_\d+\.html"))

    courses = []
    for a in links:
        # TODO: paste your working code_ + name extraction here,
        #       appending {"course_code": ..., "course_name": ...}
        href = a["href"]
        text = a.get_text(strip=True)
        course_code = re.search(r"_(\d+)\.html", href).group(1)
        course_name = re.match(r"(.+?)\s+\d+\s*unit", text).group(1)
        courses.append({
            "course_code" : course_code,
            "course_name" : course_name
        })

    index_df = pd.DataFrame(courses)
    return index_df[index_df["course_name"].isin(targets)]

In [19]:

years = [2021, 2022, 2023, 2024, 2025]

all_rows = []
for year in years:
    tdf = get_target_courses(year)
    print(year, "->", len(tdf), "courses")     # verify: expect 9 each year

    for course in tdf.itertuples():
        url = f"{BASE}/BDHSC_{year}_12_{course.course_code}.html"
        filename = f"BDHSC_{year}_{course.course_code}.html"
        # TODO: html = get_html(url, filename)
        #       all_rows.extend(parse_course_page(html, year, course.course_name, course.course_code))
        html = get_html(url,filename)
        all_rows.extend(parse_course_page(html,year,course_name=course.course_name, course_code=course.course_code))


df_all = pd.DataFrame(all_rows)
print(df_all.shape)
df_all

2021 -> 9 courses
2022 -> 9 courses
2023 -> 9 courses
2024 -> 9 courses
2025 -> 9 courses
(250, 6)


,band,percentage,candidature,year,course_name,course_code
0,6,7.16,18712,2021,Biology,15030
1,5,24.14,18712,2021,Biology,15030
2,4,34.79,18712,2021,Biology,15030
3,3,25.15,18712,2021,Biology,15030
4,2,6.68,18712,2021,Biology,15030
...,...,...,...,...,...,...
245,5,25.19,8817,2025,Physics,15330
246,4,25.75,8817,2025,Physics,15330
247,3,21.21,8817,2025,Physics,15330
248,2,13.81,8817,2025,Physics,15330


In [20]:
df_all.to_csv("../data/interim/hsc_bands_raw.csv", index=False)

In [21]:
print(index_df["course_name"].head().tolist())
# ['Aboriginal Studies 2 unit (15000)', ...]   <- aha, not clean!

['Aboriginal Studies', 'Agriculture', 'Ancient History', 'Arabic Continuers', 'Arabic Extension']
